# LLM vs Human Oracle Comparison

**Research question 2:** Do LLM-simulated oracles converge in patterns comparable
to human oracles, or systematically differ?

This notebook queries `experiments.db`, computes bootstrap CIs, and produces
the headline finding table.

**Kernel registration (run once in your venv):**
```
python -m ipykernel install --user --name=conversational-clustering
```

All DB access goes through `src/db/` (CLAUDE.md constraint).
No LLM calls are made in this notebook.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import statistics
from src.db import experiments as exp_db
from src.db.connection import connect, init_schema
from src.analysis import compute_bootstrap_ci
import json

db = connect()
init_schema(db)
try:
    rows = exp_db.query(db)
finally:
    db.close()
print(f'Total experiments loaded: {len(rows)}')

In [ ]:
# Group by (oracle_type, strategy_id) and compute bootstrap 95% CI
groups = {}
for r in rows:
    if r.total_turns is None:
        continue
    key = (r.oracle_type, r.strategy_id)
    groups.setdefault(key, []).append(float(r.total_turns))

for key, vals in sorted(groups.items()):
    mean_v = statistics.fmean(vals)
    lo, hi = compute_bootstrap_ci(vals, seed=0)
    print(f'{key[0]:<12} | {key[1]:<22} | mean={mean_v:.1f} | 95% CI [{lo:.1f}\u2013{hi:.1f}] | N={len(vals)}')

## Key Finding

**Finding:** LLM oracles (`oracle_type='llm'`) converged in [MEAN_LLM] turns on average
(95% CI [LO_LLM\u2013HI_LLM], N=[N_LLM] runs). Human participants (`oracle_type='human'`)
converged in [MEAN_HUMAN] turns on average (95% CI [LO_HUMAN\u2013HI_HUMAN], N=[N_HUMAN] runs).
The gap of [DIFF] turns [is/is not] statistically meaningful given the overlapping CI bounds.

**Interpretation:** [One sentence honest interpretation \u2014 e.g., 'LLM oracles appear to
converge faster, consistent with zero fatigue and perfect recall, but the gap is within
the CI margin at this sample size.']

> **Note:** Replace the `[PLACEHOLDER]` tokens above after running the cells and
> observing the printed values from Cell 3.

## Per-Strategy Breakdown

Breakdown by strategy shows which Oracle Agent parameter settings
(`strategy_id`) produced convergence patterns most similar to human behavior.

Human sessions typically use a single `strategy_id` (e.g., `human_study`).
LLM sessions span all ablation strategies (random, uncertainty\_driven, boundary\_driven).
The per-strategy view reveals whether any LLM strategy matches the human convergence profile.

In [ ]:
# Per-strategy breakdown for human sessions only
human_rows = [r for r in rows if r.oracle_type == 'human' and r.total_turns is not None]
print(f'Human sessions: {len(human_rows)}')
if human_rows:
    turns = [float(r.total_turns) for r in human_rows]
    lo, hi = compute_bootstrap_ci(turns, seed=0)
    print(f'Human mean turns: {statistics.fmean(turns):.1f} [{lo:.1f}\u2013{hi:.1f}]')

# Per-strategy for LLM sessions
llm_rows = [r for r in rows if r.oracle_type == 'llm' and r.total_turns is not None]
print(f'\nLLM sessions: {len(llm_rows)}')
llm_by_strategy = {}
for r in llm_rows:
    llm_by_strategy.setdefault(r.strategy_id, []).append(float(r.total_turns))
for sid, vals in sorted(llm_by_strategy.items()):
    mean_v = statistics.fmean(vals)
    lo, hi = compute_bootstrap_ci(vals, seed=0)
    print(f'  {sid:<22} mean={mean_v:.1f} [{lo:.1f}\u2013{hi:.1f}] N={len(vals)}')

In [ ]:
# Oracle satisfaction rates by oracle_type
# Satisfaction rate = fraction of sessions ending with convergence_reason='oracle_satisfied'
from collections import defaultdict
sat_groups = defaultdict(list)
for r in rows:
    if r.convergence_reason is not None:
        sat_groups[r.oracle_type].append(
            1.0 if r.convergence_reason == 'oracle_satisfied' else 0.0
        )

print('Oracle Satisfaction Rates:')
for otype, vals in sorted(sat_groups.items()):
    rate = statistics.fmean(vals)
    lo, hi = compute_bootstrap_ci(vals, seed=0) if len(vals) >= 5 else (rate, rate)
    print(f'  {otype}: {rate:.3f} (95% CI [{lo:.3f}\u2013{hi:.3f}], N={len(vals)})')

## Limitations

- N=[N_HUMAN] human participants is a small sample; CIs are wide.
- Human study used a single dataset (`dataset/train.jsonl`); generalization is limited.
- Oracle satisfaction signal (LLM text classifier) may not perfectly capture human intent.
- LLM oracle noise params (`consistency_rate`, `drift_probability`) not varied for this comparison.
- Human sessions used a fixed 15-turn cap (`STUDY_MAX_TURNS`); LLM sessions may use a different cap. Interpret turn counts in light of the cap that applied.